# Regime-Based Duration Rotation

This notebook organizes the workflow into one clean story:

1. Download macro and bond market data
2. Build regime features from the Treasury curve
3. Detect regimes with a rolling HMM
4. Translate regime probabilities into portfolio duration
5. Backtest the ETF rotation strategy

## 1. Imports and configuration

Fill in `.env` in the same folder before running this notebook.

Example:

`FRED_API_KEY=your_key_here`

In [ ]:
# Run this once if the notebook environment is missing packages.
# %pip install -r requirements.txt

In [ ]:
import os
from regime_portfolio_pipeline import (
    PipelineConfig,
    run_pipeline,
    plot_backtest,
    plot_backtest_dashboard,
    plot_regime_feature_boxplots,
    plot_regime_overview,
    plot_regime_performance,
    plot_regime_probabilities,
    plot_strategy_weights,
    regime_feature_summary,
    regime_performance_summary,
)

print('FRED_API_KEY loaded:', bool(os.getenv('FRED_API_KEY')))

config = PipelineConfig(
    start_date="2000-01-01",
    rolling_window_months=84,
    probability_smoothing_alpha=0.35,
    regime_duration_targets={0: 8.0, 1: 14.0, 2: 3.0},
)

config

## 2. Run the full pipeline

In [ ]:
results = run_pipeline(config)

features = results["features"]
regimes = results["regimes"]
strategy_details = results["strategy_details"]
return_frame = results["returns"]
metrics = results["metrics"]

metrics.round(4)

## 3. Inspect the detected regimes

In [ ]:
regimes.tail(10)

In [ ]:
plot_regime_overview(features, regimes)
plot_regime_probabilities(regimes)

## 4. Regime interpretation

In [ ]:
regime_feature_summary(features, regimes).round(3)

In [ ]:
plot_regime_feature_boxplots(features, regimes)

## 5. Portfolio construction

In [ ]:
strategy_details[["Regime_Name", "Confidence", "Target_Duration", "SHY", "IEF", "TLT"]].tail(12)

In [ ]:
plot_strategy_weights(strategy_details)

## 6. Backtest results

In [ ]:
regime_performance_summary(strategy_details).round(4)

In [ ]:
plot_regime_performance(strategy_details)

In [ ]:
plot_backtest_dashboard(return_frame)
plot_backtest(return_frame)
return_frame.tail()